# Implementação Opcional de Rede Neural do Zero — Desafio da Semana 2

Modelagem preditiva multiclasse desenvolvida sem a utilização de frameworks profundos, implementando de forma analítica o fluxo de propagação direta, retropropagação e atualização de pesos via derivadas.

In [1]:
import numpy as np

## 1. Definição da Estrutura da Rede Neural

In [2]:
class RedeNeuralDoZero:
    def __init__(self, dimensao_entrada, dimensao_oculta, dimensao_saida, semente=15):
        np.random.seed(semente)
        self.W1 = np.random.randn(dimensao_entrada, dimensao_oculta) * 0.01
        self.b1 = np.zeros((1, dimensao_oculta))
        self.W2 = np.random.randn(dimensao_oculta, dimensao_saida) * 0.01
        self.b2 = np.zeros((1, dimensao_saida))
        
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        
    def _derivada_sigmoid(self, ativacao):
        return ativacao * (1 - ativacao)
        
    def _softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
        
    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self._sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self._softmax(self.z2)
        return self.a2
        
    def backward(self, X, y_codificado, taxa_aprendizado):
        quantidade_amostras = X.shape[0]
        
        dz2 = self.a2 - y_codificado
        dW2 = np.dot(self.a1.T, dz2) / quantidade_amostras
        db2 = np.sum(dz2, axis=0, keepdims=True) / quantidade_amostras
        
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self._derivada_sigmoid(self.a1)
        dW1 = np.dot(X.T, dz1) / quantidade_amostras
        db1 = np.sum(dz1, axis=0, keepdims=True) / quantidade_amostras
        
        self.W2 -= taxa_aprendizado * dW2
        self.b2 -= taxa_aprendizado * db2
        self.W1 -= taxa_aprendizado * dW1
        self.b1 -= taxa_aprendizado * db1

## 2. Validação Operacional e Teste de Otimização

In [3]:
semente_aleatoria = 15
np.random.seed(semente_aleatoria)

quantidade_amostras = 100
dim_entrada = 10
dim_saida = 3

X_falso = np.random.randn(quantidade_amostras, dim_entrada)
y_falso = np.random.randint(0, dim_saida, quantidade_amostras)



y_falso_um_quente = np.zeros((quantidade_amostras, dim_saida))
y_falso_um_quente[np.arange(quantidade_amostras), y_falso] = 1

rede_manual = RedeNeuralDoZero(dimensao_entrada=dim_entrada, dimensao_oculta=16, dimensao_saida=dim_saida)
saida_inicial = rede_manual.forward(X_falso)
perda_inicial = -np.sum(y_falso_um_quente * np.log(saida_inicial + 1e-15)) / quantidade_amostras
rede_manual.backward(X_falso, y_falso_um_quente, taxa_aprendizado=0.1)

saida_final = rede_manual.forward(X_falso)
perda_final = -np.sum(y_falso_um_quente * np.log(saida_final + 1e-15)) / quantidade_amostras

print(f"perda Cross Entropy Inicial: {perda_inicial:.6f}")
print(f"perda Cross Entropy após uma etapa de gradiente: {perda_final:.6f}")
print(f"redução da perda atestada: {perda_inicial - perda_final:.6f}")

perda Cross Entropy Inicial: 1.098071
perda Cross Entropy após uma etapa de gradiente: 1.097297
redução da perda atestada: 0.000775
